# 03. Matrix Profile Motif Discovery

Run Matrix Profile motif discovery in regime-agnostic and regime-conditioned modes. Conditioned runs use saved Quantile/HMM regime labels from prior notebooks.


## Execution Mode

Local mode runs a small smoke test. HPC mode uses the full configured asset/frequency/window sweep.


In [1]:
EXECUTION_MODE = "auto"  # allowed: "auto", "local", "hpc"
STAGE_NAME = "03_matrix_profile_motif_discovery"
from pathlib import Path
import sys, warnings
warnings.simplefilter("default")

def find_workflow_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "hpc_config.py").exists(): return candidate
    fallback = Path(r"C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\HPC workflow\HPC_Regime_and_motif_discovery")
    if (fallback / "src" / "hpc_config.py").exists(): return fallback
    raise RuntimeError("Could not locate workflow root.")

WORKFLOW_ROOT = find_workflow_root(); SRC_DIR = WORKFLOW_ROOT / "src"
if str(SRC_DIR) not in sys.path: sys.path.insert(0, str(SRC_DIR))
from hpc_config import build_workflow_paths, detect_execution_mode, environment_info, find_project_root, get_window_lengths, load_workflow_config, output_suffix, save_config_snapshot
from io_utils import ensure_workflow_dirs, setup_stage_logger
PROJECT_ROOT = find_project_root(WORKFLOW_ROOT)
MODE = detect_execution_mode(EXECUTION_MODE , PROJECT_ROOT)
CONFIG = load_workflow_config(MODE, WORKFLOW_ROOT)
PATHS = build_workflow_paths(WORKFLOW_ROOT, PROJECT_ROOT)
ensure_workflow_dirs(PATHS)
SUFFIX = output_suffix(CONFIG)
LOGGER = setup_stage_logger(PATHS.logs, STAGE_NAME, SUFFIX)
SNAPSHOT_PATH = save_config_snapshot(CONFIG, PATHS, STAGE_NAME, MODE)
env = environment_info(PATHS)
print("Stage:", STAGE_NAME)
print("Execution mode:", MODE)
print("Project root:", PATHS.project_root)
print("Workflow root:", PATHS.workflow_root)
print("Input feature root:", PATHS.data_root)
print("Config snapshot:", SNAPSHOT_PATH)
print({"python": env["python_version"].split()[0], "platform": env["platform"], "hostname": env["hostname"], "cpu_count": env["cpu_count"], "memory_available_gb": env["memory_available_gb"], "gpu": env["gpu"]})


2026-06-17 11:45:36,105 | INFO | Log file: /home/i6404275/Final_master_thesis/HPC workflow/HPC_Regime_and_motif_discovery/results/logs/03_matrix_profile_motif_discovery.log


Stage: 03_matrix_profile_motif_discovery
Execution mode: hpc
Project root: /home/i6404275/Final_master_thesis
Workflow root: /home/i6404275/Final_master_thesis/HPC workflow/HPC_Regime_and_motif_discovery
Input feature root: /home/i6404275/Final_master_thesis/final_dataset/features
Config snapshot: /home/i6404275/Final_master_thesis/HPC workflow/HPC_Regime_and_motif_discovery/results/configs/03_matrix_profile_motif_discovery_config_snapshot.json
{'python': '3.11.15', 'platform': 'Linux-6.12.69+deb13-amd64-x86_64-with-glibc2.41', 'hostname': 'dacsgpu0002', 'cpu_count': 64, 'memory_available_gb': 732.953, 'gpu': {'cuda_visible_devices': None, 'nvidia_smi_available': False, 'nvidia_smi_output': 'No devices were found', 'cupy_available': False}}


## Discover Feature Files


In [2]:
from data_loading import discover_feature_files
from io_utils import save_table
inventory = discover_feature_files(PATHS.project_root, CONFIG)
if inventory.empty:
    raise FileNotFoundError(f"No feature parquet files discovered under {PATHS.data_root}")
inventory_for_save = inventory.copy(); inventory_for_save["path"] = inventory_for_save["path"].astype(str)
save_table(inventory_for_save, PATHS.tables / f"input_feature_inventory{SUFFIX}.csv", PATHS.tables / f"input_feature_inventory{SUFFIX}.parquet")
print("Discovered input files:")
print(inventory_for_save[["asset", "frequency", "market", "path"]].to_string(index=False))


Discovered input files:
  asset frequency market                                                                                                    path
BTCUSDT       15m crypto /home/i6404275/Final_master_thesis/final_dataset/features/crypto/BTCUSDT_15m_features_2020_2025.parquet
BTCUSDT        1h crypto  /home/i6404275/Final_master_thesis/final_dataset/features/crypto/BTCUSDT_1h_features_2020_2025.parquet
ETHUSDT       15m crypto /home/i6404275/Final_master_thesis/final_dataset/features/crypto/ETHUSDT_15m_features_2020_2025.parquet
ETHUSDT        1h crypto  /home/i6404275/Final_master_thesis/final_dataset/features/crypto/ETHUSDT_1h_features_2020_2025.parquet


## Run Matrix Profile

Both univariate and multivariate Matrix Profile runs are attempted. Failures are caught per asset/window/regime/segment and saved to `results/logs`.


In [3]:
import pandas as pd
from data_loading import apply_mode_limits, describe_feature_frame, load_feature_file
from feature_selection import choose_volatility_column, ensure_core_features, select_motif_feature_matrix
from hpc_config import get_window_lengths
from io_utils import failure_record, safe_name, save_failure_log, save_table
from matrix_profile_utils import run_multivariate_matrix_profile, run_univariate_matrix_profile
from motif_evaluation import evaluate_matrix_profile_results
from plotting_utils import plot_matrix_profile, plot_univariate_motif_overlay
from regime_utils import continuous_segments, default_regime_methods, load_regime_labels, merge_regime_labels

quantile_labels = load_regime_labels(WORKFLOW_ROOT, "quantile", SUFFIX)
hmm_labels = load_regime_labels(WORKFLOW_ROOT, "hmm", SUFFIX)
print(f"Loaded Quantile labels: {len(quantile_labels):,}")
print(f"Loaded HMM labels: {len(hmm_labels):,}")
if quantile_labels.empty and hmm_labels.empty:
    print("No saved regime labels found. Notebook 03 will run agnostic Matrix Profile only.")

all_results, all_profiles, all_feature_diagnostics = [], [], []
dataset_summaries, failures = [], []
figure_count = 0
max_figures = int(CONFIG.get("plotting", {}).get("max_figures_per_notebook", 40))
mp_cfg = CONFIG.get("matrix_profile", {})
STORE_PROFILES = CONFIG.get("active_mode") == "local"

def resolve_univariate_channel(df, channel):
    if channel in df.columns: return channel
    if channel in {"rolling_volatility_60", "rolling_vol"}: return choose_volatility_column(df)
    return None

def run_mp_slice(slice_df, asset, frequency, windows, mode, regime_method, regime_label, segment_id):
    global figure_count
    local_results = []
    try:
        prepared = ensure_core_features(slice_df, rolling_window=int(CONFIG.get("quantile", {}).get("default_rolling_window", 60))).reset_index(drop=True)
        timestamps = prepared["timestamp"].reset_index(drop=True)
        X, scaled, feature_columns, feature_diag = select_motif_feature_matrix(prepared, CONFIG)
        feature_diag.insert(0, "segment_id", segment_id); feature_diag.insert(0, "regime_label", regime_label); feature_diag.insert(0, "regime_method", regime_method); feature_diag.insert(0, "frequency", frequency); feature_diag.insert(0, "asset", asset)
        all_feature_diagnostics.append(feature_diag)
    except Exception as exc:
        failures.append(failure_record(STAGE_NAME, exc, asset, frequency, {"mode": mode, "regime_method": regime_method, "regime_label": regime_label, "segment_id": segment_id, "step": "feature_selection"}))
        return []

    for window in windows:
        if len(prepared) < 2 * int(window) + 2:
            continue
        if mp_cfg.get("run_univariate", True):
            for channel in mp_cfg.get("univariate_channels", ["close", "log_return"]):
                resolved = resolve_univariate_channel(prepared, channel)
                if not resolved: continue
                try:
                    series = pd.to_numeric(prepared[resolved], errors="coerce").ffill().bfill().fillna(0.0).to_numpy(dtype=float)
                    context = {"asset": asset, "frequency": frequency, "mode": mode, "regime_method": regime_method, "regime_label": regime_label, "segment_id": segment_id, "feature_set": resolved, "profile_type": "univariate"}
                    motifs, profile = run_univariate_matrix_profile(series, timestamps, int(window), int(mp_cfg.get("top_k", 5)), context, use_gpu=bool(mp_cfg.get("use_gpu_if_available", True)), exclusion_zone_factor=float(mp_cfg.get("exclusion_zone_factor", 0.5)))
                    local_results.append(motifs)
                    profile.insert(0, "feature_set", resolved); profile.insert(0, "window_length", int(window)); profile.insert(0, "segment_id", segment_id); profile.insert(0, "regime_label", regime_label); profile.insert(0, "regime_method", regime_method); profile.insert(0, "mode", mode); profile.insert(0, "frequency", frequency); profile.insert(0, "asset", asset)
                    if STORE_PROFILES: all_profiles.append(profile)
                    if figure_count < max_figures:
                        prefix = f"03_{safe_name(asset)}_{safe_name(frequency)}_{safe_name(mode)}_{safe_name(regime_method)}_{safe_name(regime_label)}_{safe_name(resolved)}_w{window}{SUFFIX}"
                        plot_matrix_profile(profile, motifs, f"{asset} {frequency} {mode} {resolved} w={window}", PATHS.figures / f"{prefix}_profile.png"); figure_count += 1
                    if figure_count < max_figures and not motifs.empty:
                        plot_univariate_motif_overlay(series, motifs.iloc[0], f"{asset} {frequency} motif overlay {resolved} w={window}", PATHS.figures / f"{prefix}_overlay.png"); figure_count += 1
                except Exception as exc:
                    failures.append(failure_record(STAGE_NAME, exc, asset, frequency, {"mode": mode, "regime_method": regime_method, "regime_label": regime_label, "segment_id": segment_id, "window": window, "feature_set": resolved, "profile_type": "univariate"}))
        if mp_cfg.get("run_multivariate", True):
            try:
                context = {"asset": asset, "frequency": frequency, "mode": mode, "regime_method": regime_method, "regime_label": regime_label, "segment_id": segment_id, "feature_set": "multivariate_core", "profile_type": "multivariate"}
                motifs, profile = run_multivariate_matrix_profile(X, timestamps, feature_columns, int(window), int(mp_cfg.get("top_k", 5)), context, exclusion_zone_factor=float(mp_cfg.get("exclusion_zone_factor", 0.5)))
                local_results.append(motifs)
                profile.insert(0, "feature_set", ",".join(feature_columns)); profile.insert(0, "window_length", int(window)); profile.insert(0, "segment_id", segment_id); profile.insert(0, "regime_label", regime_label); profile.insert(0, "regime_method", regime_method); profile.insert(0, "mode", mode); profile.insert(0, "frequency", frequency); profile.insert(0, "asset", asset)
                if STORE_PROFILES: all_profiles.append(profile)
                if figure_count < max_figures:
                    prefix = f"03_{safe_name(asset)}_{safe_name(frequency)}_{safe_name(mode)}_{safe_name(regime_method)}_{safe_name(regime_label)}_multivariate_w{window}{SUFFIX}"
                    plot_matrix_profile(profile, motifs, f"{asset} {frequency} {mode} multivariate w={window}", PATHS.figures / f"{prefix}_profile.png"); figure_count += 1
            except Exception as exc:
                failures.append(failure_record(STAGE_NAME, exc, asset, frequency, {"mode": mode, "regime_method": regime_method, "regime_label": regime_label, "segment_id": segment_id, "window": window, "profile_type": "multivariate"}))
    return [frame for frame in local_results if frame is not None and not frame.empty]

for meta in inventory.to_dict("records"):
    asset, frequency = meta["asset"], meta["frequency"]
    try:
        raw = load_feature_file(meta["path"])
        df = apply_mode_limits(raw, CONFIG)
        if df.empty: raise ValueError("No rows remain after execution-mode limits.")
        dataset_summaries.append(describe_feature_frame(meta, df))
        windows = get_window_lengths(frequency, CONFIG)
        print(f"\n{asset} {frequency}: rows={len(df):,}, windows={windows}")
        all_results.extend(run_mp_slice(df, asset, frequency, windows, "agnostic", "none", "all", "full_series"))
        for source, labels in [("quantile", quantile_labels), ("hmm", hmm_labels)]:
            if labels.empty: continue
            for regime_method in default_regime_methods(labels, CONFIG, source):
                merged = merge_regime_labels(df, labels, asset, frequency, regime_method)
                if "regime_label" not in merged.columns or merged["regime_label"].isna().all(): continue
                # Regime-conditioned MP uses continuous same-regime segments only; disconnected periods are never concatenated.
                segments = continuous_segments(merged["regime_label"], int(mp_cfg.get("min_segment_length", 96)), merged["timestamp"])
                max_segments = mp_cfg.get("local_max_conditioned_segments_per_group") if CONFIG.get("active_mode") == "local" else None
                if max_segments: segments = segments.groupby("regime_label", group_keys=False).head(int(max_segments))
                for segment in segments.to_dict("records"):
                    segment_df = merged.iloc[int(segment["start_pos"]): int(segment["end_pos_exclusive"])].reset_index(drop=True)
                    all_results.extend(run_mp_slice(segment_df, asset, frequency, windows, "conditioned", regime_method, segment["regime_label"], segment["segment_id"]))
    except Exception as exc:
        LOGGER.exception("Matrix Profile batch failure for %s %s", asset, frequency)
        failures.append(failure_record(STAGE_NAME, exc, asset, frequency, {"path": str(meta["path"])}))

results = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

def annotate_cross_regime_pairs(results_df, labels_df, source):
    if results_df.empty or labels_df.empty:
        return results_df
    methods = default_regime_methods(labels_df, CONFIG, source)
    if not methods:
        return results_df
    method = methods[0]
    label_subset = labels_df[labels_df["regime_method"].astype(str) == method][["asset", "frequency", "timestamp", "regime_label"]].copy()
    if label_subset.empty:
        return results_df
    label_subset["timestamp"] = pd.to_datetime(label_subset["timestamp"], utc=True, errors="coerce")
    out = results_df.copy()
    for side in [1, 2]:
        ts_col = f"motif_timestamp_{side}"
        merge_col = f"_timestamp_{side}"
        out[merge_col] = pd.to_datetime(out[ts_col], utc=True, errors="coerce")
        renamed = label_subset.rename(columns={"timestamp": merge_col, "regime_label": f"{source}_regime_{side}"})
        out = out.merge(renamed, on=["asset", "frequency", merge_col], how="left")
    out[f"{source}_regime_method_used"] = method
    out[f"{source}_cross_regime_pair"] = (out[f"{source}_regime_1"].astype(str) != out[f"{source}_regime_2"].astype(str)) & out[f"{source}_regime_1"].notna() & out[f"{source}_regime_2"].notna()
    if "cross_regime_pair" not in out.columns or source == "quantile":
        out["cross_regime_pair"] = out[f"{source}_cross_regime_pair"].astype(float)
    return out.drop(columns=[col for col in ["_timestamp_1", "_timestamp_2"] if col in out.columns])

results = annotate_cross_regime_pairs(results, quantile_labels, "quantile")
results = annotate_cross_regime_pairs(results, hmm_labels, "hmm")
profiles = pd.concat(all_profiles, ignore_index=True) if all_profiles else pd.DataFrame()
feature_diagnostics = pd.concat(all_feature_diagnostics, ignore_index=True) if all_feature_diagnostics else pd.DataFrame()
evaluation = evaluate_matrix_profile_results(results)
runtime = results.groupby(["asset", "frequency", "mode", "regime_method", "window_length", "profile_type"], dropna=False)["runtime_seconds"].sum().reset_index() if not results.empty else pd.DataFrame()

save_table(results, PATHS.motifs_matrix_profile / f"matrix_profile_motif_results{SUFFIX}.csv", PATHS.motifs_matrix_profile / f"matrix_profile_motif_results{SUFFIX}.parquet")
save_table(evaluation, PATHS.motifs_matrix_profile / f"matrix_profile_evaluation{SUFFIX}.csv", PATHS.motifs_matrix_profile / f"matrix_profile_evaluation{SUFFIX}.parquet")
save_table(runtime, PATHS.motifs_matrix_profile / f"matrix_profile_runtime{SUFFIX}.csv", PATHS.motifs_matrix_profile / f"matrix_profile_runtime{SUFFIX}.parquet")
save_table(profiles, PATHS.motifs_matrix_profile / f"matrix_profile_profiles{SUFFIX}.csv", PATHS.motifs_matrix_profile / f"matrix_profile_profiles{SUFFIX}.parquet")
save_table(feature_diagnostics, PATHS.tables / f"03_matrix_profile_feature_diagnostics{SUFFIX}.csv", PATHS.tables / f"03_matrix_profile_feature_diagnostics{SUFFIX}.parquet")
save_table(pd.DataFrame(dataset_summaries), PATHS.tables / f"03_matrix_profile_dataset_summary{SUFFIX}.csv", PATHS.tables / f"03_matrix_profile_dataset_summary{SUFFIX}.parquet")
failure_df = save_failure_log(failures, PATHS.logs / f"03_matrix_profile_failures{SUFFIX}.csv")
print("\nMatrix Profile motif rows:", len(results))
print("Evaluation rows:", len(evaluation))
print("Failures:", len(failure_df))
evaluation.head(20)


Loaded Quantile labels: 73,097,388
Loaded HMM labels: 8,121,932

BTCUSDT 15m: rows=210,280, windows=[32, 64, 128, 256]



BTCUSDT 1h: rows=52,577, windows=[24, 48, 72, 168, 336]



ETHUSDT 15m: rows=210,280, windows=[32, 64, 128, 256]



ETHUSDT 1h: rows=52,577, windows=[24, 48, 72, 168, 336]



Matrix Profile motif rows: 81987
Evaluation rows: 1384
Failures: 0


,asset,frequency,method,mode,regime_method,regime_label,window_length,feature_set,number_of_motifs,mean_motif_distance_or_score,median_motif_distance,recurrence_count,runtime_seconds,time_split_stability,cross_regime_overlap,notes
0,BTCUSDT,15m,matrix_profile,agnostic,none,all,32,close,10,0.596049,0.603702,20,234.763122,0.428571,0.6,Matrix Profile motif pairs; recurrence count t...
1,BTCUSDT,15m,matrix_profile,agnostic,none,all,32,log_return,10,1.585391,1.530427,20,71.766893,0.428571,0.3,Matrix Profile motif pairs; recurrence count t...
2,BTCUSDT,15m,matrix_profile,agnostic,none,all,32,"log_return,abs_log_return,rolling_volatility_6...",10,1.435926,1.447333,20,82187.743148,0.666667,0.3,Matrix Profile motif pairs; recurrence count t...
3,BTCUSDT,15m,matrix_profile,agnostic,none,all,32,rolling_volatility_60,10,0.126718,0.128299,20,72.998220,1.000000,0.5,Matrix Profile motif pairs; recurrence count t...
4,BTCUSDT,15m,matrix_profile,agnostic,none,all,64,close,10,0.983515,0.983485,20,75.421325,0.428571,0.6,Matrix Profile motif pairs; recurrence count t...
5,BTCUSDT,15m,matrix_profile,agnostic,none,all,64,log_return,10,3.410756,3.529147,20,79.883024,0.666667,0.0,Matrix Profile motif pairs; recurrence count t...
6,BTCUSDT,15m,matrix_profile,agnostic,none,all,64,"log_return,abs_log_return,rolling_volatility_6...",10,2.802269,2.850943,20,83244.498304,0.666667,0.3,Matrix Profile motif pairs; recurrence count t...
7,BTCUSDT,15m,matrix_profile,agnostic,none,all,64,rolling_volatility_60,10,0.311857,0.319271,20,74.501712,0.666667,0.3,Matrix Profile motif pairs; recurrence count t...
8,BTCUSDT,15m,matrix_profile,agnostic,none,all,128,close,10,1.545626,1.589199,20,78.740880,0.250000,0.6,Matrix Profile motif pairs; recurrence count t...
9,BTCUSDT,15m,matrix_profile,agnostic,none,all,128,log_return,10,6.478711,6.627243,20,74.817206,1.000000,0.3,Matrix Profile motif pairs; recurrence count t...
